In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import PowerTransformer, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, LSTM, Input, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
# CONFIGURATION
SEQUENCE_LENGTH = 24 
BATCH_SIZE = 256  
SHUFFLE_BUFFER = 15000 
EPOCHS = 50

# LOAD & PREPARE DATA
print("Loading and preparing data...")
df = pd.read_csv('/kaggle/input/datasets/prajyotr/mlpr-project-dataset/dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

# Physics Clip & Lag
df['sr_wm2'] = df['sr_wm2'].clip(0, 1200)
df['sr_lag'] = df.groupby('station_id')['sr_wm2'].shift(1).fillna(0)

# Time Features
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["hour"] = df["timestamp"].dt.hour

# Target Encoding (Critical for Regional Variability)
train_mask = df['year'] < 2025
for col in ['station_id', 'state']:
    if col in df.columns:
        mapper = df.loc[train_mask].groupby(col)['sr_wm2'].mean().to_dict()
        df[f'{col}_avg'] = df[col].map(mapper).fillna(df.loc[train_mask, 'sr_wm2'].mean())

# Cyclical Encoding
df['hour_sin'] = np.sin(df['hour'] * (2. * np.pi / 24)).astype('float32')
df['hour_cos'] = np.cos(df['hour'] * (2. * np.pi / 24)).astype('float32')

# Drop purely redundant columns 
cols_to_drop = ['at_c', 'rh_pct', 'ws_ms', 'wd_deg', 'rf_mm', 'station_id', 
                'station', 'state', 'city', 'era5_sw_down_wm2', 'hour', 'month']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

In [ ]:
# TRANSFORMATION & SCALING
print("Applying Power Transform and Scaling...")
pt = PowerTransformer(method='yeo-johnson')
df['sr_wm2_scaled'] = pt.fit_transform(df[['sr_wm2']]).astype('float32')

FEATURES = [col for col in df.columns if col not in ['sr_wm2', 'sr_wm2_scaled', 'year', 'timestamp']]
N_FEATURES = len(FEATURES)

# Split data
train_df = df[df['year'] < 2025].copy()
test_df = df[df['year'] >= 2025].copy()

scaler = StandardScaler()
train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES]).astype('float32')
test_df[FEATURES] = scaler.transform(test_df[FEATURES]).astype('float32')

# DATASET GENERATOR
def build_station_dataset(station_df, sequence_length, batch_size, shuffle=False):
    station_datasets = []
    for _, grp in station_df.groupby('station_id_avg', sort=False):
        X = grp[FEATURES].values.astype('float32')
        y = grp['sr_wm2_scaled'].values.astype('float32')
        if len(X) <= sequence_length: continue
        
        ds = tf.keras.utils.timeseries_dataset_from_array(
            data=X[:-sequence_length], targets=y[sequence_length:],
            sequence_length=sequence_length, batch_size=None, shuffle=False)
        station_datasets.append(ds)
    
    combined = station_datasets[0]
    for ds in station_datasets[1:]: combined = combined.concatenate(ds)
    
    if shuffle: combined = combined.shuffle(SHUFFLE_BUFFER)
    return combined.batch(batch_size).prefetch(tf.data.AUTOTUNE)

print("Building datasets...")
train_ds = build_station_dataset(train_df, SEQUENCE_LENGTH, BATCH_SIZE, shuffle=True)
val_ds = build_station_dataset(test_df, SEQUENCE_LENGTH, BATCH_SIZE, shuffle=False)

In [ ]:
# Model Architecture
model = Sequential()
model.add(Input(shape=(SEQUENCE_LENGTH, N_FEATURES)))

# LSTM Layer 0: Bidirectional, 96 units
model.add(Bidirectional(LSTM(96, return_sequences=True)))
model.add(BatchNormalization())

# LSTM Layer 1: 96 units
model.add(LSTM(96, return_sequences=True))
model.add(BatchNormalization())

# LSTM Layer 2: 32 units (final LSTM layer, returns flat output)
model.add(LSTM(32, return_sequences=False))
model.add(BatchNormalization())

# Dense Layer 0: 256 base units + 0.1 Dropout
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.1))

# Dense Layer 1: 128 units (halved from the previous layer as per your tuning logic)
model.add(Dense(128, activation='relu'))

# Output layer
model.add(Dense(1, activation='linear'))

# Compile with the optimized learning rate (0.001)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='huber', metrics=['mae'])

model.summary()

In [ ]:
# Training
print("\nTraining Best Model")
callbacks = [
    EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint('best_solar_lstm.keras', save_best_only=True, monitor='val_loss') # Saves automatically during training
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

model.save('final_solar_lstm.keras')
print("\nModel saved successfully to 'best_solar_lstm.keras' and 'final_solar_lstm.keras'")

# EVALUATION
print("\nEvaluation")
y_pred_scaled = model.predict(val_ds)
y_true_scaled = np.concatenate([y for _, y in val_ds], axis=0)

# Inverse transform to get actual W/m² values
y_true_real = pt.inverse_transform(y_true_scaled.reshape(-1, 1))
y_pred_real = pt.inverse_transform(y_pred_scaled)

print(f"Final R2 Score: {r2_score(y_true_real, y_pred_real):.4f}")
print(f"Final MAE: {mean_absolute_error(y_true_real, y_pred_real):.4f}")